## Deep BSDE Solver for a European Call Option with default risk

This notebook illustrates the use of a Deep BSDE solver from [E, Han and Jentzen (2017)](https://doi.org/10.1007/s40304-017-0117-6) and [Han, Jentzen and E (2018)](https://www.pnas.org/doi/abs/10.1073/pnas.1718942115) to price a European call option using the Black-Scholes model in BSDE form. 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import numpy as np
import math
import time
import pandas as pd

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

### Payoff and Network

In [ ]:
def worst_of_payoff(S, K):
    # S is of shape (M, d); take the minimum over the asset dimension
    worst_S = torch.min(S, dim=1, keepdim=True)[0]
    return torch.clamp(worst_S - K, min=0.0)

In [ ]:
# A simple fully connected network for approximating Z at each time step
class FCNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(FCNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.fc3(x)
        return x

### Pricing Function

In [ ]:
def BSDE_WorstOf_CptyDefault(S0, K, sigma, r, T, d,
                             v_h, v_l, gamma_h, gamma_l, delta,
                             M=256, N=40, hidden_dim=128,
                             max_iterations=10000, tol=1e-7, seed=42, lr=8e-3):
    start_time = time.time()
    dt = T / N
    torch.manual_seed(seed)
    slope =  (gamma_h - gamma_l) / (v_h - v_l)
    
    # Build a network for each time step; the state is the worst asset value (1D input).
    Z_nets = nn.ModuleList([FCNet(d, hidden_dim, d).to(device) for _ in range(N - 1)])
    
    # Initialize Y0 with a heuristic guess
    initial_guess = S0 * np.exp(r * T) - K
    Y0 = torch.tensor([initial_guess], device=device, requires_grad=True)
    
    optimizer = optim.Adam([Y0] + list(Z_nets.parameters()), lr=lr)
    
    loss_lst = []
    prev_Y0 = None
    iteration = 0

    pbar = tqdm(range(max_iterations), desc="Training", dynamic_ncols=True)
    for it in pbar:
        S = torch.full((M, d), S0, device=device)
        iteration += 1
        Y = Y0.repeat(M, 1)
        dW = torch.randn(M, N - 1, d, device=device) * torch.sqrt(torch.tensor(dt))
        
        # For each time step, update asset prices and the BSDE solution.
        for n in range(N - 1):
            S = S * torch.exp((r - 0.5 * sigma**2) * dt + sigma * dW[:, n:n+1]).squeeze()  
            Z = Z_nets[n](S)
            
            # BSDE drift
            piecewise_linear = F.relu(F.relu(Y - v_h) * slope + gamma_h - gamma_l) + gamma_l
            f = -((1 - delta) * piecewise_linear - r) * Y
            Y = Y - f * dt + Z * dW[:, n:n+1]
        
        # Terminal payoff for worst-of call.
        payoff_T = worst_of_payoff(S, K)
        loss = torch.mean((Y - payoff_T)**2)
        loss_lst.append(loss.item())
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Check convergence based on Y0 (the BSDE solution at time 0)
        current_Y0 = Y0.item()
        pbar.set_postfix(Y0=f"{current_Y0:.6f}", Loss=f"{loss.item():.6e}")
        
        if prev_Y0 is not None:
            rel_change = abs(current_Y0 - prev_Y0) / abs(prev_Y0)
            if rel_change < tol:
                print(f"Converged at iteration {iteration} with relative change in Y0: {rel_change:.2e}")
                break
        prev_Y0 = current_Y0

    execution_time = time.time() - start_time
    return Y0.item(), loss_lst, iteration, execution_time

In [ ]:
T = 1           # Maturity
r = 0.02
sigma = 0.2        # Volatility
S0 = 100.0         # Initial stock price
K = 0
v_h = 70 
v_l = 50
gamma_h = 0.2
gamma_l = 0.02 
delta = 2/3
d = 100

In [ ]:
Y0, loss_lst, iterations, etime = BSDE_WorstOf_CptyDefault(S0, K, sigma, r, T, d,
                             v_h, v_l, gamma_h, gamma_l, delta)

In [ ]:
Y0

In [ ]:
etime

In [ ]:
def plot_losses(losses, filename):
    epochs = np.arange(len(losses)) + 1
    plt.figure(figsize=(15, 10))
    plt.plot(epochs, losses,  color='black',  label='BSDE Training Loss')
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.yscale('log')
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
plot_losses(loss_lst, 'bsdedefloss.png')